In [26]:
import pandas as pd

df= pd.read_csv("hbl17.csv")
df.head()

,source,title,url,snippet,published_date,query,year,extracted,extraction_status
0,DAWN,The first Pakistan Banking Awards held in Kara...,https://aurora.dawn.com/news/1141468/the-first...,Habib Bank Limited also received two awards – ...,Not Found,HBL_STRICT,2016,NaN,FAIL: Date Tag Not Found
1,DAWN,HBL celebrates 70 years of Pakistan - Recent -...,https://aurora.dawn.com/news/1142498/hbl-celeb...,"Being the 'Bank of the Nation', it was inevita...",Not Found,HBL_STRICT,2017,NaN,FAIL: Date Tag Not Found
2,DAWN,Daraz records revenue of Rs 3 billion during B...,https://www.dawn.com/news/1373601,Customers were able to enjoy additional discou...,Not Found,HBL_STRICT,2017,NaN,FAIL: Date Tag Not Found
3,DAWN,A league of our own: How the PSL brand was built,http://www.dawn.com/news/1312858,In came brands such as Habib Bank Limited (HBL...,Not Found,HBL_STRICT,2017,NaN,FAIL: Date Tag Not Found
4,DAWN,Basking in the spotlight - Recent - Aurora Mag...,https://aurora.dawn.com/news/1141985/basking-i...,Company: Habib Bank Limited Agencies: J. Walte...,Not Found,HBL_STRICT,2017,NaN,FAIL: Date Tag Not Found


In [27]:
import pandas as pd
import cloudscraper
from bs4 import BeautifulSoup
import json
import numpy as np
import time
from concurrent.futures import ThreadPoolExecutor


MAX_WORKERS = 15 

def get_dawn_date(url):
    """
    Scrapes the published date from a Dawn news URL.
    Returns a tuple: (extracted_date, status_code).
    """
    # 1. Basic URL Validation
    if not isinstance(url, str) or not url.startswith('http'):
        return (None, "ERROR: Invalid URL Format")
        
    try:
        # 2. Add a short delay to be respectful to the server
        # NOTE: This delay still runs, but since 15 threads are running 
        # simultaneously, the overall time is greatly reduced.
        time.sleep(0.5) 

        scraper = cloudscraper.create_scraper()
        response = scraper.get(url, timeout=10) # Added a timeout for robustness
        
        # 3. HTTP Status Validation
        if response.status_code != 200:
            return (None, f"ERROR: HTTP {response.status_code}")
            
        html = response.text
        soup = BeautifulSoup(html, "lxml")

        # 4. Attempt Date Extraction (New Dawn format)
        tag = soup.find("span", class_="timestamp--date")
        if tag:
            date_text = tag.get_text(strip=True)
            return (date_text, "SUCCESS")

        # 5. Handle Missing Date/Tag
        return (None, "FAIL: Date Tag Not Found")

    except Exception as e:
        # Catch network errors, connection timeouts, etc.
        return (None, f"ERROR: Exception - {type(e).__name__}")


# --- Main Execution Logic ---

# 1. Load your DataFrame (Replace this line with your actual loading)
# Example: 
# df = pd.read_csv("your_source_file.csv") 
# Assuming 'df' is loaded and has a 'url' column

# Example DataFrame for testing (if you don't have one ready):
# urls = [
#     "https://www.dawn.com/news/1875080/us-envoy-discusses-human-rights-with-cm-maryam",
#     "https://www.dawn.com/news/1875069/pakistan-to-acquire-nuclear-power-reactor-from-china",
#     "https://www.dawn.com/news/9999999/this-url-is-likely-to-fail-404",
#     "https://www.dawn.com/news/1875082/lhc-seeks-report-on-power-outages",
#     # ... (add 210 URLs here for a real test)
# ]
# df = pd.DataFrame({"url": urls})


print(f"Starting concurrent scraping with {len(df)} rows and {MAX_WORKERS} workers...")
start_time = time.time()

# 2. Use ThreadPoolExecutor for concurrent scraping
# It applies the get_dawn_date function to every url in the list concurrently.
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Get all URLs from the DataFrame as a list
    url_list = df["url"].tolist()
    
    # map() runs the function across all items in url_list and returns the results in order
    results = list(executor.map(get_dawn_date, url_list))

end_time = time.time()
print(f"Scraping completed in: {end_time - start_time:.2f} seconds")

# 3. Add the results back to the DataFrame
df["extracted_dates"] = results

# 4. Split the tuple column into two separate columns
df[["extracted", "extraction_status"]] = pd.DataFrame(
    df["extracted_dates"].tolist(), index=df.index
)

# 5. Clean up the intermediate column
df.drop(columns=["extracted_dates"], inplace=True)


# --- Create and Save Two Separate Files ---

# A. DataFrame for Successfully Extracted Dates
extracted_dates_df = df[df["extraction_status"] == "SUCCESS"].copy()
print(f"\n✅ Successfully Extracted Dates: {len(extracted_dates_df)} rows")
extracted_dates_df.to_csv("successful_extractions17.0.csv", index=False)

# B. DataFrame for Missing/Failed Dates
# Includes all rows where the status is NOT 'SUCCESS'
missing_dates_df = df[df["extraction_status"] != "SUCCESS"].copy()
print(f"❌ Missing/Failed Extractions: {len(missing_dates_df)} rows")
missing_dates_df.to_csv("failed_extractions.csv", index=False)


print("\nDone! Results saved to 'successful_extractions.csv' and 'failed_extractions.csv'.")

Starting concurrent scraping with 25 rows and 15 workers...
Scraping completed in: 3.32 seconds

✅ Successfully Extracted Dates: 0 rows
❌ Missing/Failed Extractions: 25 rows

Done! Results saved to 'successful_extractions.csv' and 'failed_extractions.csv'.
